# 17. Pressure contribution and simple forecast benchmarks

Hold the compact model's feature list and 60-day window fixed. Compare cases today + trailing 7-day mean + target weekday/ZIP FE, with and without 7-day commuting pressure. This isolates the predictive contribution of pressure conditional on that local-history specification. It does not measure causal infection importation.

Same 585 regions, weekly refits, horizons 1/2/3, early/later-2021 periods as Notebook 16. Include persistence (future = today) and seasonal naive (target weekday in the previous week). Retain Full and Backward forecasts as context. Source hashes, regions, dates and outcomes must match the previous run.

No feature reselection, 2022-2023 test evaluation, outbreak warnings or invented spatial diagnostics. Selection/window choices followed earlier validation inspection, so all statistical results remain exploratory.


In [1]:
"""Validation-only feature ablations and exploratory inference for NJ onset counts.

Requires numpy/pandas. Optional matplotlib creates report figures. No GMM tests.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd

REPO=Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
SOURCE=REPO/'nj_daily_feature_outputs'/'nj_daily_features.csv.gz'

BASE=['cases_today','cases_lag_1','cases_lag_7']
HISTORY=BASE+['cases_mean_7d']
EXTENDED=HISTORY+['cases_lag_14','case_std_7d']
ACC=EXTENDED+['case_acceleration']
DYNAMICS=ACC+['case_jerk']
SPECIFICATIONS={
    'B0_basic':BASE,
    'H0_weekly_mean':HISTORY,
    'H1_history_extended':EXTENDED,
    'D1_acceleration':ACC,
    'D2_jerk':DYNAMICS,
    'P0_basic_pressure7':BASE+['nj_covid_pressure_7d'],
    'P1_rich_pressure7':DYNAMICS+['nj_covid_pressure_7d'],
    'P2_rich_pressure14':DYNAMICS+['nj_covid_pressure_14d'],
}
COMPARISONS=[
    ('B0_basic','H0_weekly_mean','weekly_mean'),
    ('H0_weekly_mean','H1_history_extended','lag14_and_volatility'),
    ('H1_history_extended','D1_acceleration','acceleration'),
    ('D1_acceleration','D2_jerk','jerk'),
    ('B0_basic','P0_basic_pressure7','pressure7_basic'),
    ('D2_jerk','P1_rich_pressure7','pressure7_rich'),
    ('D2_jerk','P2_rich_pressure14','pressure14_rich'),
    ('P1_rich_pressure7','P2_rich_pressure14','pressure14_vs_pressure7'),
]


def two_way_meat(scores,n_parameters):
    """CR1 two-way cluster sandwich meat: region + date - region/date cell."""
    N,T,p=scores.shape
    n=N*T
    if min(N,T)<=1 or n<=n_parameters:
        return np.full((p,p),np.nan)
    base=(n-1)/(n-n_parameters)
    by_region=scores.sum(axis=1)
    by_date=scores.sum(axis=0)
    cells=scores.reshape(-1,p)
    return base*(N/(N-1)*(by_region.T@by_region)+T/(T-1)*(by_date.T@by_date)-n/(n-1)*(cells.T@cells))


def fit_fe(x,y):
    N,T,p=x.shape
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm=x.mean(axis=1); ym=y.mean(axis=1)
    xc=x-xm[:,None,:]; yc=y-ym[:,None]
    scale=np.sqrt(np.mean(xc*xc,axis=(0,1)))
    scale=np.where(scale>1e-12,scale,1)
    z=xc/scale
    zf=z.reshape(-1,p)
    b,_,rank,sv=np.linalg.lstsq(zf,yc.reshape(-1),rcond=None)
    beta=b/scale; alpha=ym-xm@beta
    residual=yc-np.einsum('ntp,p->nt',z,b)
    if rank==p:
        bread=np.linalg.pinv(zf.T@zf,rcond=1e-12)
        covariance=bread@two_way_meat(z*residual[:,:,None],N+p)@bread
        variance=np.diag(covariance)/(scale*scale)
        se=np.full(p,np.nan)
        positive=np.isfinite(variance)&(variance>0)
        se[positive]=np.sqrt(variance[positive])
    else:
        se=np.full(p,np.nan)
    pvalue=np.array([math.erfc(abs(v/s)/math.sqrt(2)) if np.isfinite(s) and s>0 else np.nan for v,s in zip(beta,se)])
    condition=float(sv[0]/sv[-1]) if len(sv) and sv[-1]>0 else np.inf
    return beta,alpha,se,pvalue,int(rank),condition


def hac_mean_test(d,bandwidth=14):
    """Two-sided asymptotic test of zero mean paired DATE-level loss difference."""
    d=np.asarray(d,float)
    assert np.isfinite(d).all()
    T=len(d); L=min(int(bandwidth),T-1)
    mean=float(d.mean()); c=d-mean
    longvar=float(c@c/T)
    for lag in range(1,L+1):
        longvar+=2*(1-lag/(L+1))*float(c[lag:]@c[:-lag]/T)
    variance=longvar/T*T/(T-1)
    if variance<=0 or not np.isfinite(variance):
        return mean,np.nan,np.nan
    se=math.sqrt(variance)
    return mean,se,math.erfc(abs(mean/se)/math.sqrt(2))


def holm_adjust(p):
    p=np.asarray(p,float); adjusted=np.full(p.shape,np.nan)
    good=np.flatnonzero(np.isfinite(p)); ordered=good[np.argsort(p[good])]
    if len(ordered):
        values=np.maximum.accumulate((len(ordered)-np.arange(len(ordered)))*p[ordered])
        adjusted[ordered]=np.minimum(values,1)
    return adjusted


def block_indices(T,block_length,repetitions,seed):
    """Noncircular moving-block resampling; one date index shared by all regions/horizons."""
    assert 1<=block_length<=T
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block_length+1,size=(repetitions,math.ceil(T/block_length)))
    return (starts[:,:,None]+np.arange(block_length)[None,None,:]).reshape(repetitions,-1)[:,:T]


def residual_correlation(errors,lag):
    a=errors[:,lag:]; b=errors[:,:-lag]
    a=a-a.mean(axis=1,keepdims=True); b=b-b.mean(axis=1,keepdims=True)
    den=np.sqrt(np.sum(a*a,axis=1)*np.sum(b*b,axis=1))
    value=np.full(len(a),np.nan)
    good=den>0; value[good]=np.sum(a[good]*b[good],axis=1)/den[good]
    return value


def self_checks():
    # FWL estimates and cluster covariance versus an explicit dummy design.
    rng=np.random.default_rng(729)
    N,T,p=5,12,2
    x=rng.normal(size=(N,T,p)); y=x@np.array([.5,-.25])+np.arange(N)[:,None]+rng.normal(size=(N,T))
    beta,alpha,se,pv,rank,_=fit_fe(x,y)
    design=np.column_stack([np.repeat(np.eye(N),T,axis=0),x.reshape(-1,p)])
    full=np.linalg.lstsq(design,y.reshape(-1),rcond=None)[0]
    assert np.allclose(full[:N],alpha) and np.allclose(full[N:],beta)
    e=y.reshape(-1)-design@full
    inv=np.linalg.pinv(design.T@design)
    covariance=inv@two_way_meat((design*e[:,None]).reshape(N,T,N+p),N+p)@inv
    positive=np.diag(covariance)[N:]>0
    assert np.allclose(se[positive]**2,np.diag(covariance)[N:][positive])
    assert np.allclose(holm_adjust([.01,.04,.03]),[.03,.06,.06])
    independent=rng.normal(size=100)
    _,s,_=hac_mean_test(independent,0)
    assert np.isclose(s,np.std(independent,ddof=1)/10)



GROUPS = {
    'core_history': ['cases_today','cases_lag_1','cases_lag_7','cases_lag_14'],
    'rolling_level': ['cases_mean_7d','cases_mean_14d'],
    'volatility': ['case_std_7d','case_std_14d'],
    'dynamics': ['case_acceleration','case_jerk'],
    'local_rates': ['case_rate_today_per_100k','case_rate_7d_per_100k','case_rate_14d_per_100k'],
    'sparse_history': ['days_since_last_observed_case_censored','no_previous_observed_case'],
    'pressure': ['nj_covid_pressure_7d','nj_covid_pressure_14d'],
    'month': [f'target_month_{k}' for k in range(2,13)],
}
WEEKDAY=[f'target_weekday_{k}' for k in range(2,8)]

def independent_columns(x):
    # Training-only rank filtering. Removes algebraic duplicates and calendar
    # categories absent from the current window; not predictive feature selection.
    z=x-x.mean(axis=1,keepdims=True)
    norms=np.sqrt(np.mean(z*z,axis=(0,1)))
    selected=[]; q=[]
    for j in range(z.shape[2]):
        if norms[j]<1e-10: continue
        v=(z[:,:,j]/norms[j]).reshape(-1).copy()
        for _ in range(2):
            for u in q:v-=u*(u@v)
        length=np.linalg.norm(v)
        if length>1e-7*np.sqrt(v.size):
            selected.append(j);q.append(v/length)
    return selected


def run_pressure_check(source,output,reference_folder,window=60,bootstrap_repetitions=2000):
    """Greedy backward selection on early 2021; later 2021 checked after freezing."""
    self_checks()
    source=Path(source)
    folder=Path(output)/('pressure_check_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    folder.mkdir(parents=True)
    numeric=sum([v for k,v in GROUPS.items() if k!='month'],[])
    full=numeric+GROUPS['month']+WEEKDAY
    cols=sorted(set(numeric)-set(GROUPS['sparse_history']))
    p=pd.read_csv(source,usecols=cols+['zip','date','population','days_since_last_observed_case','pressure_definition'],dtype={'zip':'string'},parse_dates=['date'])
    assert p.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    assert not p.duplicated(['zip','date']).any()
    ids=np.array(sorted(p.zip.unique()));dates=pd.date_range(p.date.min(),p.date.max())
    def matrix(c):return p.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float)
    population=matrix('population')[:,0]
    eligible=(population>0)&np.isfinite(population)&np.isfinite(matrix('nj_covid_pressure_14d')[:,13])
    matrices={c:matrix(c)[eligible] for c in cols}
    gap=matrix('days_since_last_observed_case')[eligible]
    matrices['no_previous_observed_case']=(~np.isfinite(gap)).astype(float)
    matrices['days_since_last_observed_case_censored']=np.where(np.isfinite(gap),gap,np.arange(len(dates))[None,:]+1)
    ids=ids[eligible];n=len(ids);counts=matrices['cases_today']
    origins=np.flatnonzero((dates>=pd.Timestamp('2021-01-01'))&((dates+pd.Timedelta(days=3))<=pd.Timestamp('2021-12-31')))
    v=len(origins);actual=np.stack([counts[:,origins+h] for h in (1,2,3)])
    selection_mask=(dates[origins]+pd.Timedelta(days=3))<=pd.Timestamp('2021-06-30')
    assessment_mask=dates[origins]>=pd.Timestamp('2021-07-01')
    assert not (selection_mask&assessment_mask).any()
    # Cached sufficient statistics let every candidate refit the actual OLS
    # on its own feature subset without repeating full raw training-data scans.
    cache=[];timing=[]
    for h in (1,2,3):
        target=dates+pd.Timedelta(days=h)
        for k in range(2,8):matrices[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),(n,len(dates)))
        for k in range(2,13):matrices[f'target_month_{k}']=np.broadcast_to((target.month==k).astype(float),(n,len(dates)))
        x=np.stack([matrices[c] for c in full],axis=2)
        blocks=[]
        for first in range(0,v,7):
            block=origins[first:first+7];fit=int(block[0]);last=fit-h;train=np.arange(last-window+1,last+1)
            assert train[0]>=14 and train[-1]+h<=fit
            xt=x[:,train];yt=counts[:,train+h]
            assert np.isfinite(xt).all() and np.isfinite(x[:,block]).all()
            xm=xt.mean(axis=1);ym=yt.mean(axis=1);xc=xt-xm[:,None];yc=yt-ym[:,None]
            scale=np.sqrt((xc*xc).mean(axis=(0,1)));scale=np.where(scale>1e-10,scale,1)
            z=xc/scale;zf=z.reshape(-1,len(full))
            gram=zf.T@zf;rhs=zf.T@yc.reshape(-1)
            blocks.append((first,len(block),gram,rhs,scale,xm,ym,x[:,block]))
            timing.append({'horizon':h,'fit_date':dates[fit],'training_origin_start':dates[train[0]],'training_origin_end':dates[train[-1]],'latest_training_target':dates[train[-1]+h]})
        cache.append(blocks)
    np.seterr(invalid='raise')
    def predict(features,return_rank_log=False):
        ix=[full.index(f) for f in features]
        predictions=np.empty((3,n,v));rank_log=[]
        for hh,blocks in enumerate(cache):
            for first,length,g,rhs,scale,xm,ym,xp in blocks:
                # Ordered Gram-Schmidt in training Gram space removes only
                # nonidentified columns. Standardization is training-only.
                selected=[]
                for j in ix:
                    if g[j,j]<1e-8:continue
                    if not selected:residual=g[j,j]
                    else:
                        gg=g[np.ix_(selected,selected)];cross=g[selected,j]
                        residual=g[j,j]-cross@np.linalg.solve(gg,cross)
                    if residual>1e-8*max(g[j,j],1):selected.append(j)
                if selected:
                    b=np.linalg.solve(g[np.ix_(selected,selected)],rhs[selected])/scale[selected]
                    a=ym-xm[:,selected]@b
                    predictions[hh,:,first:first+length]=np.einsum('ntp,p->nt',xp[:,:,selected],b)+a[:,None]
                else:predictions[hh,:,first:first+length]=ym[:,None]
                if return_rank_log:rank_log.append({'horizon':hh+1,'forecast_origin':dates[origins[first]],'estimated_slopes':len(selected),'omitted':';'.join(full[j] for j in ix if j not in selected)})
        assert np.isfinite(predictions).all()
        return (predictions,rank_log) if return_rank_log else predictions
    def losses(pred,mask):
        e=np.maximum(pred,0)[:,:,mask]-actual[:,:,mask]
        return np.abs(e).mean(axis=1),(e*e).mean(axis=1),e.mean(axis=(1,2))
    # Numerical regression check: one subset must match direct within OLS.
    hh=0;first,length,g,rhs,scale,xm,ym,xp=cache[hh][0]
    check_features=['cases_today','cases_lag_1','cases_lag_7']+WEEKDAY
    check_ix=[full.index(f) for f in check_features]
    train_end=int(origins[0])-1;tr=np.arange(train_end-window+1,train_end+1)
    check_x=np.stack([matrices[f] if f not in WEEKDAY else np.broadcast_to(((dates+pd.Timedelta(days=1)).dayofweek==int(f.rsplit('_',1)[1])-1).astype(float),(n,len(dates))) for f in check_features],axis=2)
    direct_b,direct_a,*_=fit_fe(check_x[:,tr],counts[:,tr+1])
    direct=np.einsum('ntp,p->nt',check_x[:,origins[:length]],direct_b)+direct_a[:,None]
    assert np.allclose(predict(check_features)[0,:,:length],direct,atol=1e-7,rtol=1e-7)



    reference_folder=Path(reference_folder)
    reference_protocol=json.loads((reference_folder/'protocol.json').read_text(encoding='utf-8'))
    assert reference_protocol['source_sha256']==hashlib.sha256(source.read_bytes()).hexdigest()
    assert window==60
    with np.load(reference_folder/'predictions_60d.npz',allow_pickle=False) as a:
        assert np.array_equal(ids,a['zip']) and np.array_equal(dates[origins].to_numpy(),a['date'])
        assert np.array_equal(actual,a['actual_cases'])
        assert np.array_equal(selection_mask,a['selection_mask']) and np.array_equal(assessment_mask,a['assessment_mask'])
        reference_models=list(a['models']);reference_predictions=a['raw_predictions'].copy()
    with_pressure=['cases_today','cases_mean_7d','nj_covid_pressure_7d']+WEEKDAY
    without_pressure=['cases_today','cases_mean_7d']+WEEKDAY
    frozen=json.loads((reference_folder/'frozen_feature_lists.json').read_text(encoding='utf-8'))
    assert frozen['FORWARD']['selected_columns']==with_pressure
    reproduced=predict(with_pressure)
    difference=float(np.max(np.abs(reproduced-reference_predictions[reference_models.index('FORWARD')])))
    assert difference<1e-7
    no_pressure,rank_log=predict(without_pressure,True)
    persistence=np.stack([counts[:,origins] for h in (1,2,3)])
    seasonal=np.stack([counts[:,origins+h-7] for h in (1,2,3)])
    assert all((origins+h-7<=origins).all() for h in (1,2,3))
    models=['COMPACT_WITH_PRESSURE','COMPACT_NO_PRESSURE','PERSISTENCE','SEASONAL_NAIVE','FULL_REFERENCE','BACKWARD_REFERENCE']
    predictions=np.stack([reproduced,no_pressure,persistence,seasonal,
        reference_predictions[reference_models.index('FULL')],reference_predictions[reference_models.index('BACKWARD')]])
    protocol={'source':str(source.resolve()),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'reference_folder':str(reference_folder.resolve()),'window':60,'regions':n,'horizons':[1,2,3],
        'retrain_every_days':7,'feature_reselection':False,'with_pressure':with_pressure,'without_pressure':without_pressure,
        'contrast':'Only remove 7-day pressure from the frozen compact model; keep all other inputs and forecast rows identical.',
        'persistence':'C(i,t) for every horizon h','seasonal_naive':'C(i,t+h-7), target weekday in previous week',
        'periods':{'selection':'Jan-Jun 2021, original selection period','later_validation':'Jul-Dec 2021, exploratory temporal validation'},
        'comparisons':'Compact with pressure versus no pressure, persistence, seasonal naive; each period overall and horizons 1/2/3 = 24 contrasts.',
        'statistics':'Paired date-level mean absolute loss, HAC Bartlett bandwidth 14, Holm across 24 contrasts. Pointwise moving-date-block CIs, 2000 replicates; 7/28-day sensitivity.',
        'limitations':'Pressure and 60-day window chosen after earlier validation inspection. Exploratory, not causal or an untouched holdout. Fixed-forecast bootstrap does not include fitting/selection uncertainty.',
        'test_2022_2023_evaluated':False}
    (folder/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    pd.DataFrame(timing).to_csv(folder/'training_timing.csv',index=False)
    pd.DataFrame(rank_log).to_csv(folder/'no_pressure_rank_log.csv',index=False)
    metrics=[];daily=[];monthly=[];regional=[];paired=[];sensitivity=[]
    # Fixed case-volume strata derive only from 2020, never evaluation outcomes.
    volume=counts[:,dates<pd.Timestamp('2021-01-01')].mean(axis=1)
    quartile=pd.cut(pd.Series(volume).rank(pct=True,method='average'),[0,.25,.5,.75,1],labels=['low','lower_middle','upper_middle','high']).astype(str).to_numpy()
    pd.DataFrame({'zip':ids,'mean_cases_2020':volume,'volume_group_2020':quartile}).to_csv(folder/'region_cohort.csv',index=False)
    for period,mask in [('selection',selection_mask),('later_validation',assessment_mask)]:
        error=np.maximum(predictions[:,:,:,mask],0)-actual[None,:,:,mask]
        ae=np.abs(error).mean(axis=2);mse=(error*error).mean(axis=2)
        months=dates[origins[mask]].strftime('%Y-%m').to_numpy()
        for m,model in enumerate(models):
            for hh in range(3):
                e=error[m,hh]
                metrics.append({'period':period,'model':model,'horizon':hh+1,'regions':n,'forecast_dates':int(mask.sum()),
                    'MAE':float(ae[m,hh].mean()),'RMSE':float(np.sqrt(mse[m,hh].mean())),'bias':float(e.mean())})
                for day,aa,ms in zip(dates[origins[mask]],ae[m,hh],mse[m,hh]):daily.append({'period':period,'model':model,'horizon':hh+1,'date':day,'MAE':aa,'MSE':ms})
                for month in sorted(set(months)):
                    mm=months==month;em=e[:,mm]
                    monthly.append({'period':period,'model':model,'horizon':hh+1,'forecast_origin_month':month,
                        'dates':int(mm.sum()),'MAE':float(np.abs(em).mean()),'RMSE':float(np.sqrt((em*em).mean())),'bias':float(em.mean())})
                for i,z in enumerate(ids):regional.append({'period':period,'model':model,'horizon':hh+1,'zip':z,'volume_group_2020':quartile[i],
                    'MAE':float(np.abs(e[i]).mean()),'RMSE':float(np.sqrt((e[i]*e[i]).mean())),'bias':float(e[i].mean())})
        draws={b:block_indices(int(mask.sum()),b,bootstrap_repetitions,20261102+b) for b in (7,14,28)}
        for baseline in ['COMPACT_NO_PRESSURE','PERSISTENCE','SEASONAL_NAIVE']:
            b=models.index(baseline);c=0
            for horizon in ['overall',1,2,3]:
                hs=np.arange(3) if horizon=='overall' else np.array([horizon-1])
                d=(ae[b,hs]-ae[c,hs]).mean(axis=0)
                delta,se,pv=hac_mean_test(d,14)
                row={'period':period,'baseline':baseline,'candidate':'COMPACT_WITH_PRESSURE','horizon':horizon,
                    'baseline_MAE':float(ae[b,hs].mean()),'candidate_MAE':float(ae[c,hs].mean()),
                    'delta_MAE_baseline_minus_candidate':delta,'MAE_reduction_percent':100*delta/ae[b,hs].mean(),
                    'p_HAC_raw_exploratory':pv,'p_HAC_bandwidth7':hac_mean_test(d,7)[2],'p_HAC_bandwidth28':hac_mean_test(d,28)[2]}
                for length,index in draws.items():
                    low,high=np.quantile(d[index].mean(axis=1),[.025,.975])
                    sensitivity.append({'period':period,'baseline':baseline,'horizon':horizon,'block_days':length,'delta_MAE_ci95_low':low,'delta_MAE_ci95_high':high})
                    if length==14:row.update(ci95_low_pointwise=low,ci95_high_pointwise=high)
                paired.append(row)
    metrics=pd.DataFrame(metrics);paired=pd.DataFrame(paired)
    paired['p_Holm_all_24_exploratory']=holm_adjust(paired.p_HAC_raw_exploratory)
    summary=metrics.groupby(['period','model'],as_index=False)[['MAE','RMSE','bias']].mean()
    metrics.to_csv(folder/'metrics_by_horizon.csv',index=False);summary.to_csv(folder/'summary_metrics.csv',index=False)
    paired.to_csv(folder/'paired_pressure_and_benchmark_tests.csv',index=False)
    pd.DataFrame(sensitivity).to_csv(folder/'block_length_sensitivity.csv',index=False)
    pd.DataFrame(daily).to_csv(folder/'daily_losses.csv',index=False)
    pd.DataFrame(monthly).to_csv(folder/'metrics_by_month.csv',index=False)
    regional=pd.DataFrame(regional);regional.to_csv(folder/'metrics_by_region.csv',index=False)
    region_comparison=regional.loc[regional.period.eq('later_validation')].groupby(['zip','volume_group_2020','model']).MAE.mean().unstack('model').reset_index()
    region_comparison['delta_MAE_no_pressure_minus_with_pressure']=region_comparison.COMPACT_NO_PRESSURE-region_comparison.COMPACT_WITH_PRESSURE
    region_comparison.to_csv(folder/'regional_pressure_contribution.csv',index=False)
    np.savez_compressed(folder/'predictions.npz',raw_predictions=predictions,actual_cases=actual,models=np.array(models),zip=ids,date=dates[origins].to_numpy(),horizons=np.array([1,2,3]),selection_mask=selection_mask,assessment_mask=assessment_mask)
    report='# Compact FE model: pressure contribution and simple benchmarks\n\n'
    report+='Same frozen feature list, 585 regions, 60-day training, weekly refits, direct days 1/2/3. Remove only 7-day pressure; fit again, preserving weekday controls, ZIP FE and local cases today/7-day mean. No reselection. Include persistence and previous-week target weekday benchmarks. Full/Backward are context from Notebook 16, not new searches.\n\n'
    report+='## Later-2021 validation\n\n| Model | Mean MAE | Mean RMSE | Bias |\n|---|---:|---:|---:|\n'
    for r in summary.loc[summary.period.eq('later_validation')].itertuples():report+=f'| {r.model} | {r.MAE:.6f} | {r.RMSE:.6f} | {r.bias:.6f} |\n'
    report+='\n## Paired overall comparisons\n\nPositive difference favors the compact model with pressure.\n\n| Baseline | Delta MAE | 95% pointwise CI | Holm p |\n|---|---:|---|---:|\n'
    for r in paired.loc[paired.period.eq('later_validation')&paired.horizon.eq('overall')].itertuples():report+=f'| {r.baseline} | {r.delta_MAE_baseline_minus_candidate:.6f} | [{r.ci95_low_pointwise:.6f}, {r.ci95_high_pointwise:.6f}] | {r.p_Holm_all_24_exploratory:.4g} |\n'
    report+='\nMonthly results group by forecast-origin month; regional case-volume strata use 2020 only. Regional/monthly comparisons are descriptive, not additional independent hypothesis tests. No spatial Moran test without real spatial weights. These results are exploratory after earlier selection/window inspection, not final-test or causal evidence. 2022-2023 test data are not evaluated. Intervals are pointwise date-block resampling of fixed forecasts; no full-pipeline uncertainty.\n'
    (folder/'RESULTS.md').write_text(report,encoding='utf-8')
    checks={'same_source_cohort_dates_targets_as_notebook16':True,'with_pressure_predictions_reproduced':True,
        'max_reference_prediction_difference':difference,'benchmark_inputs_no_later_than_forecast_origin':True,
        'features_reselected':False,'regions':n,'test_evaluated':False,'paired_contrasts':len(paired)}
    (folder/'verification.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
    print(summary.to_string(index=False),flush=True)
    print(paired.loc[paired.period.eq('later_validation')&paired.horizon.eq('overall')].to_string(index=False),flush=True)
    print('Saved to:',folder.resolve(),flush=True)
    return summary,metrics,paired,folder


In [2]:
SOURCE = Path.cwd() / 'nj_daily_feature_outputs' / 'nj_daily_features.csv.gz'
REFERENCE = Path.cwd() / 'individual_feature_selection_outputs' / 'frozen_window_60_20261026T153858733079Z'
OUTPUT = Path.cwd() / 'pressure_validation_outputs'
summary, horizons, comparisons, RUN_FOLDER = run_pressure_check(SOURCE, OUTPUT, REFERENCE)
display(summary.loc[summary.period.eq('later_validation')])
display(horizons.loc[horizons.period.eq('later_validation')])
display(comparisons.loc[comparisons.period.eq('later_validation')])
print(RUN_FOLDER.resolve())


          period                 model      MAE     RMSE      bias
later_validation    BACKWARD_REFERENCE 1.159535 2.003030 -0.022042
later_validation   COMPACT_NO_PRESSURE 1.127037 2.008233 -0.189913
later_validation COMPACT_WITH_PRESSURE 1.164881 2.003248 -0.027737
later_validation        FULL_REFERENCE 1.159097 2.008269 -0.000615
later_validation           PERSISTENCE 1.346114 2.388163 -0.069320
later_validation        SEASONAL_NAIVE 1.383413 2.548214 -0.302397
       selection    BACKWARD_REFERENCE 1.043649 1.768703  0.118322
       selection   COMPACT_NO_PRESSURE 1.083822 1.797078  0.191890
       selection COMPACT_WITH_PRESSURE 1.040341 1.776808  0.117398
       selection        FULL_REFERENCE 1.043405 1.760428  0.123995
       selection           PERSISTENCE 1.276260 2.297659  0.038999
       selection        SEASONAL_NAIVE 1.326233 2.332011  0.150914
          period            baseline             candidate horizon  baseline_MAE  candidate_MAE  delta_MAE_baseline_minus_candida

,period,model,MAE,RMSE,bias
0,later_validation,BACKWARD_REFERENCE,1.159535,2.003030,-0.022042
1,later_validation,COMPACT_NO_PRESSURE,1.127037,2.008233,-0.189913
2,later_validation,COMPACT_WITH_PRESSURE,1.164881,2.003248,-0.027737
3,later_validation,FULL_REFERENCE,1.159097,2.008269,-0.000615
4,later_validation,PERSISTENCE,1.346114,2.388163,-0.069320
5,later_validation,SEASONAL_NAIVE,1.383413,2.548214,-0.302397


,period,model,horizon,regions,forecast_dates,MAE,RMSE,bias
18,later_validation,COMPACT_WITH_PRESSURE,1,585,181,1.112571,1.849927,-0.030218
19,later_validation,COMPACT_WITH_PRESSURE,2,585,181,1.163523,1.998826,-0.029942
20,later_validation,COMPACT_WITH_PRESSURE,3,585,181,1.218550,2.160992,-0.023051
21,later_validation,COMPACT_NO_PRESSURE,1,585,181,1.080844,1.855476,-0.172308
22,later_validation,COMPACT_NO_PRESSURE,2,585,181,1.125917,2.003873,-0.191896
23,later_validation,COMPACT_NO_PRESSURE,3,585,181,1.174351,2.165352,-0.205535
24,later_validation,PERSISTENCE,1,585,181,1.293422,2.220783,-0.038702
25,later_validation,PERSISTENCE,2,585,181,1.346999,2.395371,-0.071068
26,later_validation,PERSISTENCE,3,585,181,1.397922,2.548336,-0.098191
27,later_validation,SEASONAL_NAIVE,1,585,181,1.368806,2.524628,-0.307428


,period,baseline,candidate,horizon,baseline_MAE,candidate_MAE,delta_MAE_baseline_minus_candidate,MAE_reduction_percent,p_HAC_raw_exploratory,p_HAC_bandwidth7,p_HAC_bandwidth28,ci95_low_pointwise,ci95_high_pointwise,p_Holm_all_24_exploratory
12,later_validation,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,overall,1.127037,1.164881,-0.037844,-3.357822,1.844064e-02,3.202157e-03,5.144896e-02,-0.068026,-0.006821,9.230150e-02
13,later_validation,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,1,1.080844,1.112571,-0.031727,-2.935385,1.893406e-02,2.925481e-03,5.525755e-02,-0.058086,-0.005689,9.230150e-02
14,later_validation,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,2,1.125917,1.163523,-0.037606,-3.340028,1.915202e-02,3.397697e-03,5.266306e-02,-0.068157,-0.006704,9.230150e-02
15,later_validation,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,3,1.174351,1.218550,-0.044199,-3.763684,1.881680e-02,3.778373e-03,4.956867e-02,-0.078474,-0.007434,9.230150e-02
16,later_validation,PERSISTENCE,COMPACT_WITH_PRESSURE,overall,1.346114,1.164881,0.181233,13.463435,7.239034e-20,2.104288e-30,6.033895e-14,0.143007,0.210347,1.664978e-18
17,later_validation,PERSISTENCE,COMPACT_WITH_PRESSURE,1,1.293422,1.112571,0.180851,13.982390,1.767529e-22,1.229254e-34,3.437389e-15,0.146848,0.211794,4.242069e-21
18,later_validation,PERSISTENCE,COMPACT_WITH_PRESSURE,2,1.346999,1.163523,0.183476,13.621067,7.744128e-19,1.195957e-27,4.282020e-14,0.144602,0.210398,1.703708e-17
19,later_validation,PERSISTENCE,COMPACT_WITH_PRESSURE,3,1.397922,1.218550,0.179373,12.831383,8.160837e-17,6.774454e-25,5.441586e-12,0.137926,0.210674,1.713776e-15
20,later_validation,SEASONAL_NAIVE,COMPACT_WITH_PRESSURE,overall,1.383413,1.164881,0.218532,15.796555,3.269449e-08,3.782830e-11,2.233536e-06,0.153084,0.285012,4.904174e-07
21,later_validation,SEASONAL_NAIVE,COMPACT_WITH_PRESSURE,1,1.368806,1.112571,0.256235,18.719602,2.298574e-06,6.700999e-09,6.460797e-05,0.167093,0.325785,2.988147e-05


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\pressure_validation_outputs\pressure_check_20261026T161618775983Z


## Read first

* RESULTS.md / summary_metrics.csv: overall performance and interpretation.
* metrics_by_horizon.csv: days 1/2/3; lower average errors may hide horizon tradeoffs.
* paired_pressure_and_benchmark_tests.csv: positive baseline-minus-candidate delta favors pressure model; HAC p-values, Holm correction across 24 contrasts and pointwise date-block intervals.
* metrics_by_month.csv: stability across forecast-origin months.
* regional_pressure_contribution.csv: which regions improve; positive delta favors pressure. Descriptive, not per-region significance.
* block_length_sensitivity.csv: 7/14/28-day dependence sensitivity.

PR-AUC and warning lead time require subsequent outbreak definitions. Residual Moran's I needs real geographic weights. Prediction improvement, statistical evidence and causal interpretation are separate questions.


## Optional report figures

Requires matplotlib. PNG (300 dpi) and SVG copies are saved in the run's figures folder.


In [3]:
from pathlib import Path

def plot_pressure_check(folder):
    folder=Path(folder);dest=folder/'figures';dest.mkdir(exist_ok=True)
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    summary=pd.read_csv(folder/'summary_metrics.csv').query("period == 'later_validation'").set_index('model')
    order=['COMPACT_NO_PRESSURE','COMPACT_WITH_PRESSURE','PERSISTENCE','SEASONAL_NAIVE']
    labels=['Compact\nno pressure','Compact\nwith pressure','Persistence','Seasonal naive']
    colors=['#059669','#2563eb','#94a3b8','#64748b']
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for ax,metric in zip(axes,['MAE','RMSE']):
        values=summary.reindex(order)[metric]
        bars=ax.bar(labels,values,color=colors);ax.bar_label(bars,fmt='%.4f',padding=4)
        ax.set_title(f'Later-2021 {metric}');ax.set_ylabel('Recorded-onset cases');ax.set_ylim(0,values.max()*1.18)
        ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
    fig.suptitle('Pressure ablation and simple benchmarks: frozen compact FE model')
    fig.text(.5,.015,'585 regions | 60-day training | Weekly refits | Days 1/2/3 | Exploratory validation; no test evaluation',ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.06,1,.94])
    for ext in ['png','svg']:fig.savefig(dest/f'01_pressure_benchmark_comparison.{ext}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    months=pd.read_csv(folder/'metrics_by_month.csv').groupby(['forecast_origin_month','model']).MAE.mean().unstack()
    delta=months.COMPACT_NO_PRESSURE-months.COMPACT_WITH_PRESSURE
    fig,ax=plt.subplots(figsize=(11,4.5))
    ax.bar(delta.index,delta,color=np.where(delta>=0,'#059669','#ea580c'))
    ax.axhline(0,color='#334155',linewidth=1)
    ax.set_ylabel('MAE without pressure minus MAE with pressure')
    ax.set_title('Monthly pressure contribution: positive favors pressure')
    ax.tick_params(axis='x',rotation=40);ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
    fig.text(.5,.015,'Forecast-origin months; horizons averaged equally | Descriptive monthly stability, not independent significance tests',ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.05,1,1])
    for ext in ['png','svg']:fig.savefig(dest/f'02_monthly_pressure_contribution.{ext}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    return dest


In [4]:
try:
    print(plot_pressure_check(RUN_FOLDER))
except ImportError:
    print('Optional plots need matplotlib: %pip install matplotlib')


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\pressure_validation_outputs\pressure_check_20261026T161618775983Z\figures


## Reopen the completed run without refitting

Run this cell independently to read the saved results.


In [5]:
from pathlib import Path
import pandas as pd
EXISTING = Path.cwd() / 'pressure_validation_outputs' / 'pressure_check_20261026T160907020618Z'
display(pd.read_csv(EXISTING / 'summary_metrics.csv'))
display(pd.read_csv(EXISTING / 'paired_pressure_and_benchmark_tests.csv'))
print((EXISTING / 'RESULTS.md').read_text(encoding='utf-8'))


,period,model,MAE,RMSE,bias
0,later_validation,BACKWARD_REFERENCE,1.159535,2.003030,-0.022042
1,later_validation,COMPACT_NO_PRESSURE,1.127037,2.008233,-0.189913
2,later_validation,COMPACT_WITH_PRESSURE,1.164881,2.003248,-0.027737
3,later_validation,FULL_REFERENCE,1.159097,2.008269,-0.000615
4,later_validation,PERSISTENCE,1.346114,2.388163,-0.069320
5,later_validation,SEASONAL_NAIVE,1.383413,2.548214,-0.302397
6,selection,BACKWARD_REFERENCE,1.043649,1.768703,0.118322
7,selection,COMPACT_NO_PRESSURE,1.083822,1.797078,0.191890
8,selection,COMPACT_WITH_PRESSURE,1.040341,1.776808,0.117398
9,selection,FULL_REFERENCE,1.043405,1.760428,0.123995


,period,baseline,candidate,horizon,baseline_MAE,candidate_MAE,delta_MAE_baseline_minus_candidate,MAE_reduction_percent,p_HAC_raw_exploratory,p_HAC_bandwidth7,p_HAC_bandwidth28,ci95_low_pointwise,ci95_high_pointwise,p_Holm_all_24_exploratory
0,selection,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,overall,1.083822,1.040341,0.043481,4.011820,1.195438e-02,1.237962e-03,2.923384e-02,0.016924,0.083391,9.230150e-02
1,selection,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,1,1.074038,1.034670,0.039368,3.665423,1.153769e-02,1.165304e-03,2.949537e-02,0.015677,0.074340,9.230150e-02
2,selection,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,2,1.086167,1.043002,0.043165,3.974053,1.185195e-02,1.235010e-03,2.908789e-02,0.017141,0.082581,9.230150e-02
3,selection,COMPACT_NO_PRESSURE,COMPACT_WITH_PRESSURE,3,1.091259,1.043349,0.047910,4.390342,1.278886e-02,1.434264e-03,2.964199e-02,0.017711,0.092145,9.230150e-02
4,selection,PERSISTENCE,COMPACT_WITH_PRESSURE,overall,1.276260,1.040341,0.235920,18.485247,5.480583e-06,1.073847e-08,2.670374e-04,0.137109,0.305386,5.480583e-05
5,selection,PERSISTENCE,COMPACT_WITH_PRESSURE,1,1.271708,1.034670,0.237038,18.639368,3.312724e-06,7.027429e-09,1.859324e-04,0.139171,0.301726,3.975269e-05
6,selection,PERSISTENCE,COMPACT_WITH_PRESSURE,2,1.280169,1.043002,0.237167,18.526189,3.513044e-06,4.869420e-09,2.035402e-04,0.139194,0.304326,3.975269e-05
7,selection,PERSISTENCE,COMPACT_WITH_PRESSURE,3,1.276904,1.043349,0.233555,18.290707,1.664067e-05,5.328568e-08,5.410739e-04,0.128176,0.310025,1.497660e-04
8,selection,SEASONAL_NAIVE,COMPACT_WITH_PRESSURE,overall,1.326233,1.040341,0.285893,21.556743,4.512097e-10,9.340725e-16,8.080245e-07,0.205251,0.365589,8.572984e-09
9,selection,SEASONAL_NAIVE,COMPACT_WITH_PRESSURE,1,1.339652,1.034670,0.304982,22.765782,9.572478e-11,9.049554e-17,3.327772e-07,0.223808,0.391818,1.914496e-09


# Compact FE model: pressure contribution and simple benchmarks

Same frozen feature list, 585 regions, 60-day training, weekly refits, direct days 1/2/3. Remove only 7-day pressure; fit again, preserving weekday controls, ZIP FE and local cases today/7-day mean. No reselection. Include persistence and previous-week target weekday benchmarks. Full/Backward are context from Notebook 16, not new searches.

## Later-2021 validation

| Model | Mean MAE | Mean RMSE | Bias |
|---|---:|---:|---:|
| BACKWARD_REFERENCE | 1.159535 | 2.003030 | -0.022042 |
| COMPACT_NO_PRESSURE | 1.127037 | 2.008233 | -0.189913 |
| COMPACT_WITH_PRESSURE | 1.164881 | 2.003248 | -0.027737 |
| FULL_REFERENCE | 1.159097 | 2.008269 | -0.000615 |
| PERSISTENCE | 1.346114 | 2.388163 | -0.069320 |
| SEASONAL_NAIVE | 1.383413 | 2.548214 | -0.302397 |

## Paired overall comparisons

Positive difference favors the compact model with pressure.

| Baseline | Delta MAE | 95% pointwise CI | Holm p |
|---|---:|---|---:|
| COMPAC